In [2]:
%%sh
pip install tiktoken

In [4]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [9]:
sample_text = "Hello there, how are you?"
tokenizer.encode(sample_text)

[15496, 612, 11, 703, 389, 345, 30]

## What's happening here?

This is doing *Byte-Pair* encoding.

I don't really care about the specifics here... it's just.. implemented.. somehow..

## How to think about this..

Need a map $T: \text{[str]} \rightarrow \text{[num]}$ to be able to work with text.

So the idea is to NOT encode the alphabet and do character indices.

What I mean is, we *could* do char2idx = {'a': 0, 'b': 1, ...} and an inverse idx2char to get text from numeric ML model output:

In [17]:
char2idx = {}
idx2char = {}
alphabet = "abcdefghijklmnopqrstuvwxyz"

for i, char in enumerate(alphabet+alphabet.upper()+",.!? "): # AND numbers, etc..
    char2idx[char] = i
    idx2char[i] = char

In [18]:
def alphabetical_tokenizer(text: str):
    return [char2idx[char] for char in text]

alphabetical_tokenizer(sample_text)

[33,
 4,
 11,
 11,
 14,
 56,
 19,
 7,
 4,
 17,
 4,
 52,
 56,
 7,
 14,
 22,
 56,
 0,
 17,
 4,
 56,
 24,
 14,
 20,
 55]

In [20]:
len(alphabetical_tokenizer(sample_text)), len(tokenizer.encode(sample_text))

(25, 7)

## There you go
Byte-Pair encoding uses MUCH less bits AND gives us a larger vocabulary/alphabet (the GPT-2 vocabulary is 50257, good luck writing that by hand..)

## Next, embeddings
I always got confused about tokens, embeddings, encodings, etc.

Actually, it's straightforward:
1. **Encode** *text* into *tokens* (i.e. "Hello, you!" turns into [15496, 11, 345, 0])
    - What do those numbers *mean*? $\rightarrow$ can be words, or two words (if they're frequent) or words with special characters, ... it's just a more optimal way to handle the information present in text.
    - So we have numbers now, but we're not done. These numbers are too high and generally all over the place ('Hello' is near 15000 while '!' is 0). Neural networks won't like that. They need *normalized* inputs, ideally within a certain mean and standard deviation (e.g. $\mu=0, \sigma=1$). This leads us to the next point:
2. **Embed** the *tokens* (i.e. [15496, 11, 345, 0] turns into [-0.5835, -1.2272, -1.2691,  0.8038] (w. embedding dimension=1))

    - This is just a lookup table! (sorta, the specific "lookup" mapping is learned, like the rest of the neural network)
    - It's like our idx2char lookup table except it's token2embedding (or perhaps we'd call it nat2real, because it makes us go from natural numbers (0, 1, ..) to real numbers (something like 0.3423553 or -1.5562)
    - See code below for how it actually looks with the PyTorch embedding layer.
    - We can adjust the size for embeddings (before training). This means we can get e.g. TWO of those lists like the one above for ONE input token list. Or even more.. this can capture much more data.. 

In [23]:
tokenizer.encode('Hello, you!')

[15496, 11, 345, 0]

In [47]:
import torch.nn as nn
import torch

embedding_layer = nn.Embedding(num_embeddings=50257, embedding_dim=2)

embedding_layer(torch.tensor(tokenizer.encode('Hello, you!'))).T

tensor([[-0.0075, -1.2355,  0.5954,  0.2619],
        [-0.8459, -2.2466, -0.2466,  1.5204]], grad_fn=<PermuteBackward0>)

## Results
Now we're ready to feed data into a network. All you need is a dataset of tokens (I think encoding raw text *during* training is a terrible idea). 

## nvm, postional encoding
We need positional encoding. Currently there's no info for the network to learn relative positions in a given sequence.

It's done by just using another nn.Embedding except it uses the context_length for num_embeddings parameter, not the vocab_size.